In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [5]:
df = pd.read_excel("datasets/sample_-_superstore.xls")

In [6]:
analysis_date = pd.Timestamp("2026-10-24")

def calculate_baseline(data):
    total_sales = data["Sales"].sum()
    total_profit = data["Profit"].sum()

    return {
        "Rows": len(data),
        "Total Sales": total_sales,
        "Total Profit": total_profit,
        "Profit Margin (%)": (
            total_profit / total_sales * 100
            if total_sales != 0 else None
        ),
        "Total Orders": data["Order ID"].nunique(),
        "Total Quantity": data["Quantity"].sum()
    }

# Full dataset
full_results = calculate_baseline(df)

# Orders up to the analysis date
historical_df = df[df["Order Date"] <= analysis_date]
historical_results = calculate_baseline(historical_df)

comparison = pd.DataFrame({
    "Full Dataset": full_results,
    "As of 2026-10-24": historical_results
})

print(comparison.round(2))

                   Full Dataset  As of 2026-10-24
Rows                   10194.00           9195.00
Total Sales          2326534.35        2115813.28
Total Profit          292296.81         273117.69
Profit Margin (%)         12.56             12.91
Total Orders            5111.00           4588.00
Total Quantity         38654.00          34787.00


In [7]:
# Analyze profitability across discount levels

discount_analysis = (
    historical_df
    .groupby("Discount")
    .agg(
        Orders=("Order ID", "nunique"),
        Rows=("Order ID", "size"),
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Quantity=("Quantity", "sum")
    )
    .reset_index()
)

discount_analysis["Profit Margin (%)"] = (
    discount_analysis["Profit"]
    / discount_analysis["Sales"]
    * 100
)

discount_analysis = discount_analysis.sort_values("Discount")

discount_analysis.round(2)

,Discount,Orders,Rows,Sales,Profit,Quantity,Profit Margin (%)
0,0.00,2433,4443,999084.60,296383.55,16913,29.67
1,0.10,86,91,51190.48,8451.75,353,16.51
2,0.15,46,47,25037.44,1407.58,173,5.62
3,0.20,2193,3336,712341.21,84652.96,12453,11.88
4,0.30,192,208,96218.60,-9754.22,782,-10.14
5,0.32,24,24,13880.87,-2337.82,98,-16.84
6,0.40,165,185,106128.86,-20744.63,689,-19.55
7,0.45,9,10,4763.10,-2073.11,39,-43.52
8,0.50,59,61,49747.94,-16120.13,228,-32.40
9,0.60,123,132,6128.19,-5212.27,501,-85.05


In [8]:
category_discount_analysis = (
    historical_df
    .groupby(["Category", "Discount"])
    .agg(
        Rows=("Order ID", "size"),
        Orders=("Order ID", "nunique"),
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Quantity=("Quantity", "sum")
    )
    .reset_index()
)

category_discount_analysis["Profit Margin (%)"] = (
    category_discount_analysis["Profit"]
    / category_discount_analysis["Sales"]
    * 100
)

category_discount_analysis = category_discount_analysis.sort_values(
    ["Category", "Discount"]
)

category_discount_analysis.round(2)

,Category,Discount,Rows,Orders,Sales,Profit,Quantity,Profit Margin (%)
0,Furniture,0.00,785,686,233690.17,53738.95,3085,23.00
1,Furniture,0.10,73,68,43455.38,6533.59,293,15.04
2,Furniture,0.15,47,46,25037.44,1407.58,173,5.62
3,Furniture,0.20,568,504,201647.81,6317.22,2071,3.13
4,Furniture,0.30,203,188,92462.29,-10080.25,770,-10.90
5,Furniture,0.32,24,24,13880.87,-2337.82,98,-16.84
6,Furniture,0.40,67,64,39438.80,-14132.41,253,-35.83
7,Furniture,0.45,10,9,4763.10,-2073.11,39,-43.52
8,Furniture,0.50,52,51,20581.84,-12604.67,204,-61.24
9,Furniture,0.60,130,121,6074.23,-5182.00,492,-85.31


In [9]:
product_profitability = (
    historical_df
    .groupby(
        ["Category", "Sub-Category", "Product Name"],
        as_index=False
    )
    .agg(
        Rows=("Order ID", "size"),
        Orders=("Order ID", "nunique"),
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Quantity=("Quantity", "sum"),
        Avg_Discount=("Discount", "mean")
    )
)

product_profitability["Profit Margin (%)"] = (
    product_profitability["Profit"]
    / product_profitability["Sales"]
    * 100
)

loss_making_products = (
    product_profitability[
        product_profitability["Profit"] < 0
    ]
    .sort_values("Profit")
    .head(15)
)

loss_making_products.round(2)

,Category,Sub-Category,Product Name,Rows,Orders,Sales,Profit,Quantity,Avg_Discount,Profit Margin (%)
1608,Technology,Machines,Cubify CubeX 3D Printer Double Head Print,3,3,11099.96,-8879.97,9,0.53,-80.00
1626,Technology,Machines,Lexmark MX611dhe Monochrome Laser Printer,4,4,16829.90,-4589.97,18,0.40,-27.27
350,Furniture,Tables,Chromcraft Bull-Nose Wood Oval Conference Tabl...,5,5,9917.64,-2876.12,27,0.28,-29.00
723,Office Supplies,Binders,GBC DocuBind P400 Electric Binding System,6,6,17965.07,-1878.17,27,0.45,-10.45
1607,Technology,Machines,Cisco TelePresence System EX90 Videoconferenci...,1,1,22638.48,-1811.08,6,0.50,-8.00
1426,Office Supplies,Supplies,Martin Yale Chadless Opener Electric Letter Op...,5,5,14990.58,-1332.50,20,0.12,-8.89
344,Furniture,Tables,Bush Advantage Collection Racetrack Conference...,6,6,7763.04,-1281.11,26,0.34,-16.50
325,Furniture,Tables,Balt Solid Wood Round Tables,4,4,6518.75,-1201.06,19,0.20,-18.42
343,Furniture,Tables,Bretford “Just In Time” Height-Adjustable Mult...,3,3,3965.30,-1081.07,13,0.30,-27.26
24,Furniture,Bookcases,O'Sullivan 4-Shelf Bookcase in Odessa Pine,6,6,1403.37,-1006.55,24,0.47,-71.72


In [10]:
subcategory_analysis = (
    historical_df
    .groupby(["Category", "Sub-Category"])
    .agg(
        Rows=("Order ID", "size"),
        Orders=("Order ID", "nunique"),
        Products=("Product Name", "nunique"),
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Quantity=("Quantity", "sum"),
        Avg_Discount=("Discount", "mean")
    )
    .reset_index()
)

subcategory_analysis["Profit Margin (%)"] = (
    subcategory_analysis["Profit"]
    / subcategory_analysis["Sales"]
    * 100
)

subcategory_analysis = subcategory_analysis.sort_values(
    "Profit"
)

subcategory_analysis.round(2)

,Category,Sub-Category,Rows,Orders,Products,Sales,Profit,Quantity,Avg_Discount,Profit Margin (%)
3,Furniture,Tables,293,282,56,184795.12,-15822.42,1119,0.26,-8.56
0,Furniture,Bookcases,212,208,49,108827.77,-3115.69,804,0.22,-2.86
12,Office Supplies,Supplies,176,173,36,43645.85,-1235.62,603,0.08,-2.83
8,Office Supplies,Fasteners,206,203,34,8119.84,2264.01,873,0.08,27.88
9,Office Supplies,Labels,339,320,70,11965.88,5238.61,1303,0.07,43.78
5,Office Supplies,Art,738,676,157,25330.87,6122.06,2749,0.07,24.17
7,Office Supplies,Envelopes,234,229,44,15269.56,6435.45,841,0.08,42.15
15,Technology,Machines,108,105,62,176914.68,7997.60,414,0.30,4.52
2,Furniture,Furnishings,898,818,185,84245.84,12709.86,3403,0.14,15.09
4,Office Supplies,Appliances,427,413,97,93248.17,16107.50,1544,0.16,17.27


In [11]:
product_risk_by_subcategory = (
    product_profitability
    .groupby(["Category", "Sub-Category"])
    .agg(
        Total_Products=("Product Name", "nunique"),
        Loss_Making_Products=(
            "Profit",
            lambda x: (x < 0).sum()
        ),
        Profitable_Products=(
            "Profit",
            lambda x: (x > 0).sum()
        ),
        Total_Product_Loss=(
            "Profit",
            lambda x: x[x < 0].sum()
        )
    )
    .reset_index()
)

product_risk_by_subcategory["Loss-Making Product (%)"] = (
    product_risk_by_subcategory["Loss_Making_Products"]
    / product_risk_by_subcategory["Total_Products"]
    * 100
)

product_risk_by_subcategory = (
    product_risk_by_subcategory
    .sort_values("Total_Product_Loss")
)

product_risk_by_subcategory.round(2)

,Category,Sub-Category,Total_Products,Loss_Making_Products,Profitable_Products,Total_Product_Loss,Loss-Making Product (%)
15,Technology,Machines,62,22,40,-20460.84,35.48
3,Furniture,Tables,56,41,15,-19557.98,73.21
0,Furniture,Bookcases,49,31,18,-6833.77,63.27
1,Furniture,Chairs,88,22,66,-5080.68,25.00
11,Office Supplies,Storage,131,38,92,-4275.27,29.01
6,Office Supplies,Binders,211,52,159,-4247.15,24.64
2,Furniture,Furnishings,185,25,160,-2637.59,13.51
12,Office Supplies,Supplies,36,11,25,-2084.69,30.56
16,Technology,Phones,188,26,162,-1718.51,13.83
4,Office Supplies,Appliances,97,15,82,-1073.81,15.46


In [12]:

# Select loss-making products in the three priority sub-categories

priority_products = product_profitability[
    product_profitability["Sub-Category"].isin(
        ["Tables", "Machines", "Bookcases"]
    )
    & (product_profitability["Profit"] < 0)
].copy()

# Sort products from the largest loss to the smallest loss
priority_products = priority_products.sort_values("Profit")

# Display the 20 products with the largest losses
priority_products[
    [
        "Category",
        "Sub-Category",
        "Product Name",
        "Rows",
        "Orders",
        "Sales",
        "Profit",
        "Avg_Discount",
        "Profit Margin (%)"
    ]
].head(20).round(2)


,Category,Sub-Category,Product Name,Rows,Orders,Sales,Profit,Avg_Discount,Profit Margin (%)
1608,Technology,Machines,Cubify CubeX 3D Printer Double Head Print,3,3,11099.96,-8879.97,0.53,-80.00
1626,Technology,Machines,Lexmark MX611dhe Monochrome Laser Printer,4,4,16829.90,-4589.97,0.40,-27.27
350,Furniture,Tables,Chromcraft Bull-Nose Wood Oval Conference Tabl...,5,5,9917.64,-2876.12,0.28,-29.00
1607,Technology,Machines,Cisco TelePresence System EX90 Videoconferenci...,1,1,22638.48,-1811.08,0.50,-8.00
344,Furniture,Tables,Bush Advantage Collection Racetrack Conference...,6,6,7763.04,-1281.11,0.34,-16.50
325,Furniture,Tables,Balt Solid Wood Round Tables,4,4,6518.75,-1201.06,0.20,-18.42
343,Furniture,Tables,Bretford “Just In Time” Height-Adjustable Mult...,3,3,3965.30,-1081.07,0.30,-27.26
24,Furniture,Bookcases,O'Sullivan 4-Shelf Bookcase in Odessa Pine,6,6,1403.37,-1006.55,0.47,-71.72
332,Furniture,Tables,"Bevis Oval Conference Table, Walnut",7,7,6420.11,-944.75,0.31,-14.72
1652,Technology,Machines,Zebra GK420t Direct Thermal/Thermal Transfer P...,1,1,703.71,-938.28,0.70,-133.33


In [17]:

from pathlib import Path

output_dir = Path(r"D:\Retail Business Performance & Profitability\datasets")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "shopsphere_historical.csv"

powerbi_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Exported rows: {len(powerbi_df):,}")
print(f"Unique orders: {powerbi_df['Order ID'].nunique():,}")
print(f"Saved to: {output_path.resolve()}")


Exported rows: 9,195
Unique orders: 4,588
Saved to: D:\Retail Business Performance & Profitability\datasets\shopsphere_historical.csv
